In [2]:
import pandas as pd
import os
from pathlib import Path

In [3]:
BASE_DIR = Path.cwd().parent

file_path = BASE_DIR / "data" / "cleaned_data" / "cleaned_data.csv"

df = pd.read_csv(file_path, index_col=0)

In [4]:
df.index = pd.to_datetime(df.index)

In [5]:
df.head()

,Children apprehended and placed in CBP custody*,Children in CBP custody,Children transferred out of CBP custody,Children in HHS Care,Children discharged from HHS Care
2023-01-12,33.0,53.0,34.0,6566.0,436.0
2023-01-13,NaN,NaN,NaN,NaN,NaN
2023-01-14,NaN,NaN,NaN,NaN,NaN
2023-01-15,NaN,NaN,NaN,NaN,NaN
2023-01-16,NaN,NaN,NaN,NaN,NaN


### Problem Statement no: 1 (Total care system load)

In [6]:
df.rename(columns={"Children in CBP custody": "total load on CBP"}, inplace=True)

In [7]:
df.rename(columns={"Children in HHS Care": "total load on HHS"}, inplace=True)

In [8]:
# Total System Load = CBP Custody + HHS Care
df['total_system_load'] = df['total load on CBP'] + df['total load on HHS']


In [9]:
print(df['total_system_load'].describe())


count      720.000000
mean      6232.769444
std       2918.458078
min       2002.000000
25%       2500.750000
50%       6633.500000
75%       8249.500000
max      11762.000000
Name: total_system_load, dtype: float64


### Problem Statement:2 (Balance between inflow and out-flow of children)

In [10]:
# Balance between inflow and out-flow of children in CBP custody
df['CBP Custody net change'] = df['Children transferred out of CBP custody'] - df['Children apprehended and placed in CBP custody*']

# So, when -> [Children transferred out of CBP custody > Children apprehended and placed in CBP custody*] :- positive value 
# outflow is greater than inflow 
# [Children transferred out of CBP custody < Children apprehended and placed in CBP custody*] :- negative value 
# inflow is greater than outflow

In [11]:
df['CBP Custody net change']

2023-01-12    1.0
2023-01-13    NaN
2023-01-14    NaN
2023-01-15    NaN
2023-01-16    NaN
             ... 
2025-12-17    4.0
2025-12-18   -5.0
2025-12-19    NaN
2025-12-20    NaN
2025-12-21    5.0
Name: CBP Custody net change, Length: 1075, dtype: float64

In [12]:
print(df['CBP Custody net change'].count())
print(df.shape)

720
(1075, 7)


In [13]:
print((df['CBP Custody net change']>0).sum())
print((df['CBP Custody net change']<0).sum())
print((df['CBP Custody net change']==0).sum())

552
149
19


In [14]:
# Balance between inflow and out-flow of children in HHS care
df['HHS Net Flow'] = df['Children discharged from HHS Care'] - df['Children transferred out of CBP custody'] 
# So, when -> [Children discharged from HHS Care > Children transferred out of CBP custody] :- positive value 
# outflow is greater than inflow 
# [Children discharged from HHS Care < Children transferred out of CBP custody] :- negative value 
# inflow is greater than outflow

In [15]:
df['HHS Net Flow']

2023-01-12    402.0
2023-01-13      NaN
2023-01-14      NaN
2023-01-15      NaN
2023-01-16      NaN
              ...  
2025-12-17     -1.0
2025-12-18     10.0
2025-12-19      NaN
2025-12-20      NaN
2025-12-21      3.0
Name: HHS Net Flow, Length: 1075, dtype: float64

In [16]:
print((df['HHS Net Flow']>0).sum())
print((df['HHS Net Flow']<0).sum())
print((df['HHS Net Flow']==0).sum())
print(df['HHS Net Flow'].count())

475
238
7
720


In [17]:
df.head()

,Children apprehended and placed in CBP custody*,total load on CBP,Children transferred out of CBP custody,total load on HHS,Children discharged from HHS Care,total_system_load,CBP Custody net change,HHS Net Flow
2023-01-12,33.0,53.0,34.0,6566.0,436.0,6619.0,1.0,402.0
2023-01-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-01-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Problem Statement:3 (Capacity stress and relief periods)

In [18]:
def daily_status(row, column):
    value = row[column]
    if pd.isna(value):
        return "No Data"
    return "Relief" if value >= 0 else "Stress"

In [19]:
df['CBP daily status'] = df.apply(
    daily_status,
    axis=1,
    column='CBP Custody net change'
)

In [20]:
df['HHS daily status'] = df.apply(
    daily_status,
    axis=1,
    column='HHS Net Flow'
)

In [21]:
df.head()

,Children apprehended and placed in CBP custody*,total load on CBP,Children transferred out of CBP custody,total load on HHS,Children discharged from HHS Care,total_system_load,CBP Custody net change,HHS Net Flow,CBP daily status,HHS daily status
2023-01-12,33.0,53.0,34.0,6566.0,436.0,6619.0,1.0,402.0,Relief,Relief
2023-01-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No Data,No Data
2023-01-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No Data,No Data
2023-01-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No Data,No Data
2023-01-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No Data,No Data


### Problem Statement: 4 (Sustainability of care delivery over time)

### Trend & Temporal Analysis

In [22]:
df = df.rename(columns={
    'Children apprehended and placed in CBP custody*': 'apprehended',
    'total load on CBP': 'cbp_load',
    'Children transferred out of CBP custody': 'transferred_out',
    'total load on HHS': 'hhs_load',
    'Children discharged from HHS Care': 'discharged'
})

In [32]:
print(df.shape)
df.head()

(1075, 10)


,apprehended,cbp_load,transferred_out,hhs_load,discharged,total_system_load,CBP Custody net change,HHS Net Flow,CBP daily status,HHS daily status
2023-01-12,33.0,53.0,34.0,6566.0,436.0,6619.0,1.0,402.0,Relief,Relief
2023-01-13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No Data,No Data
2023-01-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No Data,No Data
2023-01-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No Data,No Data
2023-01-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No Data,No Data


In [23]:
def build_trends(df):
    daily = df.copy()
    weekly = df.resample('W').agg({
        'apprehended': 'sum',
        'transferred_out': 'sum',
        'discharged': 'sum',
        'hhs_load': 'last',   # snapshot columns: take end-of-period value, don't sum
        'cbp_load': 'last'
    })
    monthly = df.resample('ME').agg({
        'apprehended': 'sum',
        'transferred_out': 'sum',
        'discharged': 'sum',
        'hhs_load': 'last',
        'cbp_load': 'last'
    })
    return daily, weekly, monthly

In [24]:
daily, weekly, monthly = build_trends(df)


In [28]:
print('Daily shape:', daily.shape)
print('Weekly shape:', weekly.shape)
print('Monthly shape:', monthly.shape)
print("Monthly\n \n \n",monthly.head())
print("Daily\n \n \n",daily.head())
print("Weekly\n \n \n",weekly.head())

Daily shape: (1075, 10)
Weekly shape: (154, 5)
Monthly shape: (36, 5)
Monthly
 
 
             apprehended  transferred_out  discharged  hhs_load  cbp_load
2023-01-31        247.0            276.0      1856.0    7803.0      36.0
2023-02-28       1886.0           2637.0      5183.0    7896.0     216.0
2023-03-31       2479.0           3399.0      5375.0    8158.0     256.0
2023-04-30       3129.0           3967.0      5668.0    8352.0     243.0
2023-05-31       2746.0           3622.0      6256.0    7053.0      80.0
Daily
 
 
             apprehended  cbp_load  transferred_out  hhs_load  discharged  \
2023-01-12         33.0      53.0             34.0    6566.0       436.0   
2023-01-13          NaN       NaN              NaN       NaN         NaN   
2023-01-14          NaN       NaN              NaN       NaN         NaN   
2023-01-15          NaN       NaN              NaN       NaN         NaN   
2023-01-16          NaN       NaN              NaN       NaN         NaN   

           

In [44]:
monthly['discharge_offset_ratio'] = monthly['discharged'] / monthly['transferred_out']


In [46]:
print(monthly['discharge_offset_ratio'].describe())
print("\n")
print("Months where system was NOT keeping up (ratio < 1.0):", (monthly['discharge_offset_ratio'] < 1.0).sum())
print("Months where system WAS keeping up (ratio >= 1.0):", (monthly['discharge_offset_ratio'] >= 1.0).sum())

count    36.000000
mean      1.681439
std       1.218858
min       0.813492
25%       0.881713
50%       1.295039
75%       1.880968
max       6.724638
Name: discharge_offset_ratio, dtype: float64


Months where system was NOT keeping up (ratio < 1.0): 13
Months where system WAS keeping up (ratio >= 1.0): 23


In [41]:
def flag_high_load_periods(df, column='hhs_load', min_run_days=6):
    threshold = df[column].quantile(0.75)  # top 25% of historical load counts as "high"
    is_high = df[column] > threshold

    # find runs of consecutive True values
    run_id = (is_high != is_high.shift()).cumsum()  # increments every time the streak breaks
    run_lengths = is_high.groupby(run_id).transform('sum')

    df['sustained_high_load'] = is_high & (run_lengths >= min_run_days)
    return df

In [42]:
DataFrame_Periods = flag_high_load_periods(df)

In [47]:
print(df['sustained_high_load'].sum(), 'days flagged as sustained high load')


0 days flagged as sustained high load


### Pressure & Stress Identification

In [48]:
def add_rolling_averages(df, column='hhs_load'):
    df[f'{column}_7d_avg'] = df[column].rolling(7, min_periods=1).mean()
    df[f'{column}_14d_avg'] = df[column].rolling(14, min_periods=1).mean()
    return df

In [49]:
avg = add_rolling_averages(df)

In [50]:
df[['hhs_load', 'hhs_load_7d_avg', 'hhs_load_14d_avg']].head(10)


,hhs_load,hhs_load_7d_avg,hhs_load_14d_avg
2023-01-12,6566.0,6566.0,6566.0
2023-01-13,NaN,6566.0,6566.0
2023-01-14,NaN,6566.0,6566.0
2023-01-15,NaN,6566.0,6566.0
2023-01-16,NaN,6566.0,6566.0
2023-01-17,NaN,6566.0,6566.0
2023-01-18,NaN,6566.0,6566.0
2023-01-19,NaN,NaN,6566.0
2023-01-20,NaN,NaN,6566.0
2023-01-21,NaN,NaN,6566.0


In [51]:
def add_volatility(df, column='hhs_load', window=7):
    df[f'{column}_volatility'] = df[column].rolling(window, min_periods=1).std()
    return df

In [52]:
Volatility = add_volatility(df)

In [53]:
print(df['hhs_load_volatility'].describe())


count    1061.000000
mean       96.581732
std        90.111740
min         2.302173
25%        21.993181
50%        77.258764
75%       137.981883
max       616.105348
Name: hhs_load_volatility, dtype: float64


In [54]:
def detect_strain_windows(df, flow_column='net_outflow', min_run_days=5):
    is_stress = df[flow_column] < 0  # remember: your convention, negative = stress

    run_id = (is_stress != is_stress.shift()).cumsum()
    run_lengths = is_stress.groupby(run_id).transform('sum')

    df['in_strain_window'] = is_stress & (run_lengths >= min_run_days)
    return df

In [55]:
Window = detect_strain_windows(df, flow_column='HHS Net Flow', min_run_days=5)

In [56]:
print(df['in_strain_window'].sum(), 'days flagged inside a sustained strain window')


25 days flagged inside a sustained strain window


In [57]:
strain_days_per_month = df['in_strain_window'].resample('ME').sum()

In [58]:
backlog_accumulation_rate = df[df['in_strain_window']]['HHS Net Flow'].resample('ME').mean()

In [59]:
print("Strain days per month:")
print(strain_days_per_month)
print()
print("Avg backlog accumulation rate during strain (per month):")
print(backlog_accumulation_rate)

Strain days per month:
2023-01-31    0
2023-02-28    0
2023-03-31    0
2023-04-30    0
2023-05-31    0
2023-06-30    0
2023-07-31    0
2023-08-31    0
2023-09-30    0
2023-10-31    0
2023-11-30    0
2023-12-31    0
2024-01-31    0
2024-02-29    0
2024-03-31    0
2024-04-30    8
2024-05-31    2
2024-06-30    0
2024-07-31    0
2024-08-31    5
2024-09-30    0
2024-10-31    0
2024-11-30    0
2024-12-31    5
2025-01-31    0
2025-02-28    0
2025-03-31    5
2025-04-30    0
2025-05-31    0
2025-06-30    0
2025-07-31    0
2025-08-31    0
2025-09-30    0
2025-10-31    0
2025-11-30    0
2025-12-31    0
Freq: ME, Name: in_strain_window, dtype: int64

Avg backlog accumulation rate during strain (per month):
2024-04-30   -103.0
2024-05-31   -102.0
2024-06-30      NaN
2024-07-31      NaN
2024-08-31    -47.8
2024-09-30      NaN
2024-10-31      NaN
2024-11-30      NaN
2024-12-31    -76.6
2025-01-31      NaN
2025-02-28      NaN
2025-03-31     -7.2
Freq: ME, Name: HHS Net Flow, dtype: float64


In [60]:
df.to_csv("sorted_data.csv")